# Дз №2 Ислам Закиров БИВТ-22-16

In [15]:
import torch
import torch.nn.functional as F

def fetch_greedy_decoding(model, tokenizer, input_prompt="", max_tokens=1000, device="cuda"):
    encoded_input = tokenizer(input_prompt, return_tensors="pt").to(device)
    generated_tokens = encoded_input.input_ids
    model.to(device)

    max_tokens += encoded_input.input_ids.shape[1]
    next_token = torch.tensor(0).to(device)

    while next_token.item() != tokenizer.eos_token_id and generated_tokens.shape[1] < max_tokens:
        with torch.no_grad():
            model_output = model(input_ids=generated_tokens)
            logits = model_output.logits

        last_token_logits = logits[:, -1, :]
        next_token = torch.argmax(last_token_logits, dim=-1)

        generated_tokens = torch.cat([generated_tokens, next_token.unsqueeze(0)], dim=-1)

    return generated_tokens.view(-1)


def fetch_sampling(model, tokenizer, input_prompt="", max_tokens=1000, device="cuda"):
    encoded_input = tokenizer(input_prompt, return_tensors="pt").to(device)
    generated_tokens = encoded_input.input_ids
    model.to(device)

    max_tokens += encoded_input.input_ids.shape[1]
    next_token = torch.tensor(0).to(device)

    while next_token.item() != tokenizer.eos_token_id and generated_tokens.shape[1] < max_tokens:
        with torch.no_grad():
            model_output = model(input_ids=generated_tokens)
            logits = model_output.logits

        probabilities = torch.softmax(logits[:, -1, :], dim=-1)
        next_token = torch.multinomial(probabilities, num_samples=1)

        generated_tokens = torch.cat([generated_tokens, next_token], dim=-1)

    return generated_tokens.view(-1)



def fetch_temperature_sampling(model, tokenizer, input_prompt="", max_tokens=1000, temperature=1.0, device="cuda"):
    encoded_input = tokenizer(input_prompt, return_tensors="pt").to(device)
    generated_tokens = encoded_input.input_ids
    model.to(device)

    max_tokens += encoded_input.input_ids.shape[1]
    next_token = torch.tensor(0).to(device)

    while next_token.item() != tokenizer.eos_token_id and generated_tokens.shape[1] < max_tokens:
        with torch.no_grad():
            model_output = model(input_ids=generated_tokens)
            logits = model_output.logits

        adjusted_logits = logits[:, -1, :] / temperature
        probabilities = torch.softmax(adjusted_logits, dim=-1)
        next_token = torch.multinomial(probabilities, num_samples=1)

        generated_tokens = torch.cat([generated_tokens, next_token], dim=-1)

    return generated_tokens.view(-1)



def fetch_nucleus_sampling(model, tokenizer, input_prompt="", max_tokens=1000, temperature=1.0, top_p=1.0, device='cuda'):
    assert 0 < top_p <= 1

    encoded_input = tokenizer(input_prompt, return_tensors="pt").to(device)
    generated_tokens = encoded_input.input_ids
    model.to(device)

    max_tokens += encoded_input.input_ids.shape[1]
    next_token = torch.tensor(0).to(device)

    while next_token.item() != tokenizer.eos_token_id and generated_tokens.shape[1] < max_tokens:
        with torch.no_grad():
            model_output = model(input_ids=generated_tokens)
            logits = model_output.logits[:, -1, :]

        adjusted_logits = logits / temperature
        probabilities = torch.softmax(adjusted_logits, dim=-1).squeeze()

        sorted_indices = torch.argsort(probabilities, descending=True)
        cumulative_probs = torch.cumsum(probabilities[sorted_indices], dim=-1)
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
        sorted_indices_to_remove[0] = 0

        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        probabilities[indices_to_remove] = 0

        if probabilities.sum() == 0:
            break

        probabilities /= probabilities.sum()
        next_token = torch.multinomial(probabilities, num_samples=1)

        generated_tokens = torch.cat([generated_tokens, next_token.unsqueeze(0)], dim=-1)

    return generated_tokens.view(-1)

def beam_search(model, tokenizer, prompt="", num_beams=3, length_penalty=1, device='cpu'):
    tokenized_prompt = tokenizer(prompt, return_tensors="pt")
    generated_ids = tokenized_prompt.input_ids
    model = model.to(device)

    next_token_id = torch.tensor(0)
    candidates = []
    finished_candidates = []

    while len(finished_candidates) < num_beams:
      if not candidates and not finished_candidates:
        with torch.no_grad():
          outputs = model(input_ids=generated_ids.to(device))
          logits = outputs.logits

        log_probs = F.log_softmax(logits[0][-1], dim=0)
        k_candidates = torch.topk(log_probs, num_beams)

        candidates_tokens = k_candidates.indices.unsqueeze(-1).detach().cpu()
        candidates_probs = k_candidates.values.detach().cpu().tolist()

        candidates = [[token, score] for token, score in zip(candidates_tokens, candidates_probs) if token != tokenizer.eos_token_id]
        finished_candidates = [[token, score] for token, score in zip(candidates_tokens, candidates_probs) if token == tokenizer.eos_token_id]

      candidates2 = []
      for i, candidate in enumerate(candidates):
        token = candidate[0]
        score = candidate[1]

        tokenized_input = torch.cat([generated_ids, token.unsqueeze(0)], -1).cuda()

        with torch.no_grad():
          outputs = model(input_ids=tokenized_input)
          logits = outputs.logits

        log_probs = F.log_softmax(logits[0][-1], dim=0)
        k_candidates = torch.topk(log_probs, num_beams)
        candidates_tokens_temp = k_candidates.indices.unsqueeze(-1).detach().cpu()
        candidates_probs_temp = k_candidates.values.detach().cpu()

        candidates_temp = [[torch.cat([token, token1], dim=-1), score + score1.item()] for token1, score1 in zip(candidates_tokens_temp, candidates_probs_temp)]

        for cndt in candidates_temp:
          if cndt[0][-1] == tokenizer.eos_token_id:
            finished_candidates.append(cndt)
          else:
            candidates2.append(cndt)

      candidates = sorted(candidates2, key=lambda candidate: candidate[1] / len(candidate[0]) ** length_penalty, reverse=True)[:num_beams+1]

    return sorted(finished_candidates, key=lambda candidate: candidate[1] / len(candidate[0]) ** length_penalty, reverse=True)

## Первая задача

In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


if __name__ == "__main__":
    device = "cuda"

    model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct').eval()
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')

    hedgehog_prompt = '<|im_start|>system\nYou are a storyteller. Generate a story based on user message.<|im_end|>\n<|im_start|>user\nGenerate me a short story about a tiny hedgehog named Sonic.<|im_end|>\n<|im_start|>assistant\n'
    json_prompt = '<|im_start|>system\nYou are a JSON machine. Generate a JSON with format {"contractor": string with normalized contractor name, "sum": decimal, "currency": string with uppercased 3-letter currency code} based on user message.<|im_end|>\n<|im_start|>user\nTransfer 100 rubles and 50 kopeck to Mike<|im_end|>\n<|im_start|>assistant\n'

    hedgehog_story = fetch_greedy_decoding(model, tokenizer, input_prompt=hedgehog_prompt, max_tokens=1000, device=device)
    hedgehog_story_text = tokenizer.decode(hedgehog_story, skip_special_tokens=True)

    json_response = fetch_greedy_decoding(model, tokenizer, input_prompt=json_prompt, max_tokens=1000, device=device)
    json_response_text = tokenizer.decode(json_response, skip_special_tokens=True)

    print("Сказка про Соника:")
    print(hedgehog_story_text)
    print("\nСгенерированный JSON:")
    print(json_response_text)

Сказка про Соника:
system
You are a storyteller. Generate a story based on user message.
user
Generate me a short story about a tiny hedgehog named Sonic.
assistant
Once upon a time, in a small, cozy village nestled in the heart of the forest, there lived a tiny hedgehog named Sonic. Sonic was a curious and adventurous creature, always eager to explore the world around him. One day, while wandering through the forest, Sonic stumbled upon a hidden cave.

Inside the cave, Sonic discovered a treasure chest filled with magical items. As he opened the chest, he was amazed to see that the items were not just ordinary, but enchanted. Sonic was thrilled to find that he could use the items to help others in need.

From that day on, Sonic became a hero in the village. He used his magical powers to help people in need, and soon, the village was filled with people who were grateful for the help they received from Sonic.

Sonic's story became a legend, and people from all over the village would tel

## Вторая задача

In [14]:
if __name__ == "__main__":
    device = "cuda"

    model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct').eval()
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')

    hedgehog_prompt = '<|im_start|>system\nYou are a storyteller. Generate a story based on user message.<|im_end|>\n<|im_start|>user\nGenerate me a short story about a tiny hedgehog named Sonic.<|im_end|>\n<|im_start|>assistant\n'
    json_prompt = '<|im_start|>system\nYou are a JSON machine. Generate a JSON with format {"contractor": string with normalized contractor name, "sum": decimal, "currency": string with uppercased 3-letter currency code} based on user message.<|im_end|>\n<|im_start|>user\nTransfer 100 rubles and 50 kopeck to Mike<|im_end|>\n<|im_start|>assistant\n'

    hedgehog_story_greedy = fetch_greedy_decoding(model, tokenizer, input_prompt=hedgehog_prompt, max_tokens=1000, device=device)
    hedgehog_story_text_greedy = tokenizer.decode(hedgehog_story_greedy, skip_special_tokens=True)

    json_response_greedy = fetch_greedy_decoding(model, tokenizer, input_prompt=json_prompt, max_tokens=1000, device=device)
    json_response_text_greedy = tokenizer.decode(json_response_greedy, skip_special_tokens=True)

    print("Сказка про Соника (жадное декодирование):")
    print(hedgehog_story_text_greedy)
    print("\nСгенерированный JSON (жадное декодирование):")
    print(json_response_text_greedy)

    hedgehog_story_sampled = fetch_sampling(model, tokenizer, input_prompt=hedgehog_prompt, max_tokens=1000, device=device)
    hedgehog_story_text_sampled = tokenizer.decode(hedgehog_story_sampled, skip_special_tokens=True)

    json_response_sampled = fetch_sampling(model, tokenizer, input_prompt=json_prompt, max_tokens=1000, device=device)
    json_response_text_sampled = tokenizer.decode(json_response_sampled, skip_special_tokens=True)

    print("\nСказка про Соника (сэмплинг):")
    print(hedgehog_story_text_sampled)
    print("\nСгенерированный JSON (сэмплинг):")
    print(json_response_text_sampled)


Сказка про Соника (жадное декодирование):
system
You are a storyteller. Generate a story based on user message.
user
Generate me a short story about a tiny hedgehog named Sonic.
assistant
Once upon a time, in a small, cozy village nestled in the heart of the forest, there lived a tiny hedgehog named Sonic. Sonic was a curious and adventurous creature, always eager to explore the world around him. One day, while wandering through the forest, Sonic stumbled upon a hidden cave.

Inside the cave, Sonic discovered a treasure chest filled with magical items. As he opened the chest, he was amazed to see that the items were not just ordinary, but enchanted. Sonic was thrilled to find that he could use the items to help others in need.

From that day on, Sonic became a hero in the village. He used his magical powers to help people in need, and soon, the village was filled with people who were grateful for the help they received from Sonic.

Sonic's story became a legend, and people from all ove

## Третья задача

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if __name__ == "__main__":
    device = "cuda"

    model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct').eval()
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')

    hedgehog_prompt = '<|im_start|>system\nYou are a storyteller. Generate a story based on user message.<|im_end|>\n<|im_start|>user\nGenerate me a short story about a tiny hedgehog named Sonic.<|im_end|>\n<|im_start|>assistant\n'
    json_prompt = '<|im_start|>system\nYou are a JSON machine. Generate a JSON with format {"contractor": string with normalized contractor name, "sum": decimal, "currency": string with uppercased 3-letter currency code} based on user message.<|im_end|>\n<|im_start|>user\nTransfer 100 rubles and 50 kopeck to Mike<|im_end|>\n<|im_start|>assistant\n'

    hedgehog_story_greedy = fetch_greedy_decoding(model, tokenizer, input_prompt=hedgehog_prompt, max_tokens=1000, device=device)
    hedgehog_story_text_greedy = tokenizer.decode(hedgehog_story_greedy, skip_special_tokens=True)

    json_response_greedy = fetch_greedy_decoding(model, tokenizer, input_prompt=json_prompt, max_tokens=1000, device=device)
    json_response_text_greedy = tokenizer.decode(json_response_greedy, skip_special_tokens=True)

    print("Сказка про Соника (жадное декодирование):")
    print(hedgehog_story_text_greedy)
    print("\nСгенерированный JSON (жадное декодирование):")
    print(json_response_text_greedy)

    hedgehog_story_sampled = fetch_sampling(model, tokenizer, input_prompt=hedgehog_prompt, max_tokens=1000, device=device)
    hedgehog_story_text_sampled = tokenizer.decode(hedgehog_story_sampled, skip_special_tokens=True)

    json_response_sampled = fetch_sampling(model, tokenizer, input_prompt=json_prompt, max_tokens=1000, device=device)
    json_response_text_sampled = tokenizer.decode(json_response_sampled, skip_special_tokens=True)

    print("\nСказка про Соника (сэмплинг):")
    print(hedgehog_story_text_sampled)
    print("\nСгенерированный JSON (сэмплинг):")
    print(json_response_text_sampled)

    temperatures = [0.001, 0.1, 0.5, 1.0, 10.0]
    for temp in temperatures:
        hedgehog_story_temp = fetch_temperature_sampling(model, tokenizer, input_prompt=hedgehog_prompt, max_tokens=1000, temperature=temp, device=device)
        hedgehog_story_text_temp = tokenizer.decode(hedgehog_story_temp, skip_special_tokens=True)

        json_response_temp = fetch_temperature_sampling(model, tokenizer, input_prompt=json_prompt, max_tokens=1000, temperature=temp, device=device)
        json_response_text_temp = tokenizer.decode(json_response_temp, skip_special_tokens=True)

        print(f"\nСказка про Соника (сэмплинг с температурой {temp}):")
        print(hedgehog_story_text_temp)
        print(f"\nСгенерированный JSON (сэмплинг с температурой {temp}):")
        print(json_response_text_temp)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Сказка про Соника (жадное декодирование):
system
You are a storyteller. Generate a story based on user message.
user
Generate me a short story about a tiny hedgehog named Sonic.
assistant
Once upon a time, in a small, cozy village nestled in the heart of the forest, there lived a tiny hedgehog named Sonic. Sonic was a curious and adventurous creature, always eager to explore the world around him. One day, while wandering through the forest, Sonic stumbled upon a hidden cave.

Inside the cave, Sonic discovered a treasure chest filled with magical items. As he opened the chest, he was amazed to see that the items were not just ordinary, but enchanted. Sonic was thrilled to find that he could use the items to help others in need.

From that day on, Sonic became a hero in the village. He used his magical powers to help people in need, and soon, the village was filled with people who were grateful for the help they received from Sonic.

Sonic's story became a legend, and people from all ove

## Четвёртая задача

In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if __name__ == "__main__":
    device = "cuda"

    model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct').eval()
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')

    hedgehog_prompt = '<|im_start|>system\nYou are a storyteller. Generate a story based on user message.<|im_end|>\n<|im_start|>user\nGenerate me a short story about a tiny hedgehog named Sonic.<|im_end|>\n<|im_start|>assistant\n'
    json_prompt = '<|im_start|>system\nYou are a JSON machine. Generate a JSON with format {"contractor": string with normalized contractor name, "sum": decimal, "currency": string with uppercased 3-letter currency code} based on user message.<|im_end|>\n<|im_start|>user\nTransfer 100 rubles and 50 kopeck to Mike<|im_end|>\n<|im_start|>assistant\n'
    encoding = tokenizer(hedgehog_prompt, return_tensors="pt")
    json_encodding = tokenizer(json_prompt, return_tensors="pt")

    params = [(1, 0.9), (1, 0.15), (0.5, 0.9), (0.5, 0.15)]
    nucleus_sonic_texts = []
    nucleus_json_texts = []

    for (temperature, top_p) in params:
        sonic_tokens = fetch_nucleus_sampling(model, tokenizer, input_prompt=hedgehog_prompt,
                                               max_tokens=1000, temperature=temperature, top_p=top_p, device=device)
        sonic_text = tokenizer.decode(sonic_tokens[encoding.input_ids.shape[1]:], skip_special_tokens=True)
        nucleus_sonic_texts.append(sonic_text)

        json_tokens = fetch_nucleus_sampling(model, tokenizer, input_prompt=json_prompt,
                                              max_tokens=1000, temperature=temperature, top_p=top_p, device=device)
        json_text = tokenizer.decode(json_tokens[json_encodding.input_ids.shape[1]:], skip_special_tokens=True)
        nucleus_json_texts.append(json_text)

    print("Сгенерированные тексты про Соника:")
    i = 0
    for text in nucleus_sonic_texts:
        print(params[i])
        i += 1
        print(text)

    j = 0
    print("\nСгенерированные JSON:")
    for text in nucleus_json_texts:
        print(params[j])
        j += 1
        print(text)


Сгенерированные тексты про Соника:
(1, 0.9)
Once upon a time, in the small, sleepy town of Whisper Hollow, there lived a tiny hedgehog named Sonic. Sonic was no ordinary creature; he was the smallest living being, with scales that shimmered like the stars at night. His fur was soft and green, and he could often be found lounging on a branch of a nearby oak tree, stashing his things in his pouch hidden in the bark of the trunk.

One crisp autumn evening, as the leaves began to fall and the stars began to twinkle, Sonic heard a commotion near the house. The doors creaked open, and out came a curious-looking child, chasing the light from his old boots. The child was named Bloom, and his laughter rang out across the town, startling the tiny hedgehog. It laughed back, but not without a hint of mischief.

Bloom eventually came to its senses and let Sonic sit on his shoulder. He was curious to know more about this peculiar newcomer, and soon Sonic found himself playing with him. Bloom liked S

## Пятая задача

In [19]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if __name__ == "__main__":
    device = "cuda"

    model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct').eval()
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')

    hedgehog_prompt = '<|im_start|>system\nYou are a storyteller. Generate a story based on user message.<|im_end|>\n<|im_start|>user\nGenerate me a short story about a tiny hedgehog named Sonic.<|im_end|>\n<|im_start|>assistant\n'
    json_prompt = '<|im_start|>system\nYou are a JSON machine. Generate a JSON with format {"contractor": string with normalized contractor name, "sum": decimal, "currency": string with uppercased 3-letter currency code} based on user message.<|im_end|>\n<|im_start|>user\nTransfer 100 rubles and 50 kopeck to Mike<|im_end|>\n<|im_start|>assistant\n'

    params = [(1, 1.0), (4, 1.0), (4, 0.5), (4, 2.0), (8, 1.0)]

    beam_sonic_texts = []
    beam_json_texts = []

    for (beams, lp) in params:
        print(beams, lp)
        sonic_text = beam_search(model, tokenizer, prompt=hedgehog_prompt, num_beams=beams,
                                 length_penalty=lp, device=device)
        sonic_text = tokenizer.decode(sonic_text[0][0])
        beam_sonic_texts.append(sonic_text)

        json_text = beam_search(model, tokenizer, prompt=json_prompt, num_beams=beams,
                                length_penalty=lp, device=device)
        json_text = tokenizer.decode(json_text[0][0])
        beam_json_texts.append(json_text)

    print("Сгенерированные тексты про Соника:")
    i = 0
    for text in beam_sonic_texts:
        print(params[i])
        i += 1
        print(text)

    j = 0
    print("\nСгенерированные JSON:")
    for text in beam_json_texts:
        print(params[j])
        j += 1
        print(text)

1 1.0
4 1.0
4 0.5
4 2.0
8 1.0
Сгенерированные тексты про Соника:
(1, 1.0)
Once upon a time, in a small, cozy village nestled in the heart of the forest, there lived a tiny hedgehog named Sonic. Sonic was a curious and adventurous creature, always eager to explore the world around him. One day, while wandering through the forest, Sonic stumbled upon a hidden cave.

Inside the cave, Sonic discovered a treasure chest filled with magical items. As he opened the chest, he was amazed to see that the items were not just ordinary, but enchanted. Sonic was thrilled to find that he could use the items to help others in need.

From that day on, Sonic became a hero in the village. He used his magical powers to help people in need, and soon, the village was filled with people who were grateful for the help they received from Sonic.

Sonic's story became a legend, and people from all over the village would tell stories about him. Sonic's adventures and his magic helped to bring joy and hope to the p